# FINS notebook 14 — does a symmetry found by FINS make an emulator generalise? (exp72; Colab, GPU)

Registered in `README_EXPERIMENTS.md` (exp72, 13.10) before any of these runs. For Kuramoto–Sivashinsky, KdV and
Benjamin–Ono, seeds 42–44, the same stepper recipe is trained six times on data with constant backgrounds within ±0.5:
- `none`: as is;
- `fins`, `oracle`, `lienlsd`, `shift`: half of the training windows moved by a random element of a set of
  translations and background shifts, u(x) → u(x − s ξ(t)) + s φ(t): the part of that kind of the span FINS reported
  for the learned operators (`fins`), of LieNLSD's (`lienlsd`), the true translation and Galilean boost (`oracle`),
  translation and a constant added without its drift (`shift`, the naive augmentation); the spans come with the
  repository (`results/utility`);
- `wide`: as is, on data with backgrounds within ±3.

Every stepper is scored on the evaluation trajectories and on their exact Galilean boosts to backgrounds ±1 and ±2.

About 2–3 hours on a T4. Cell 2 runs everything in order (KS first), skips what is done and resumes an interrupted
training from its `.partial` file: after a disconnect, rerun cell 1 and cell 2. Cell 3 shows progress; cell 4 zips the
small outputs to `fins_nb14.zip` (upload that one to the chat; it can be uploaded after any equation is complete).
Drive needs about 1 GB free.

Before running, upload the newest **`fins_repo_full.zip`** from the chat to My Drive (root or one folder down).

In [ ]:
# 1. setup: Drive, the newest repository zip (with exp72), neuraloperator, pysindy, checks, GPU, Drive space
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['FINS_DATA_CACHE'] = '/content/drive/MyDrive/fins_runs/results/data_cache'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
RUNS = os.path.join(R, 'utility_runs')
os.makedirs(os.path.join(RUNS, 'logs'), exist_ok=True)


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_exp72(z):
    with zipfile.ZipFile(z) as zz:
        names = zz.namelist()
        return ('fins_repo/experiments/exp72_utility.py' in names
                and b'np.sqrt(b @ G @ b)' in zz.read('fins_repo/experiments/exp72_utility.py')
                and 'fins_repo/experiments/results/utility/spans_kdv_s44.npz' in names
                and b'SCALE' in zz.read('fins_repo/experiments/ks.py'))


ZIP_PATH = newest('fins_repo_full*.zip', has_exp72)
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive has exp72: upload the newest zip from the chat')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator', 'pysindy'], check=False)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU (or better), then rerun this cell')
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive: {free / 2 ** 30:.2f} GB free of {total / 2 ** 30:.1f} GB (needs about 1 GB)')
EQS, SEEDS = ('ks', 'kdv', 'bo'), (42, 43, 44)
ARMS = ('none', 'fins', 'oracle', 'lienlsd', 'shift', 'wide')


def run(log_name, cmd, done):
    if done():
        return True
    log = os.path.join(RUNS, 'logs', log_name + '.log')
    t0 = time.time()
    with open(log, 'a') as fh:
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        r = subprocess.call([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    lines = open(log).read().splitlines()
    keys = ('part in W', 'saved', 'empty', 'in:', 'Error', 'error')
    keep = [l.strip() for l in lines if any(k in l for k in keys)] if r == 0 else lines[-8:]
    print(log_name, 'exit', r, f'{time.time() - t0:.0f} s', *[k[:170] for k in keep[-4:]], sep='\n   ', flush=True)
    return r == 0


def f(name):
    return os.path.join(RUNS, name)

In [ ]:
# 2. everything, in order (KS, then KdV, then BO; resumable): six steppers per equation and seed, then the evaluation
for pde in EQS:
    for s in SEEDS:
        a = ['--pde', pde, '--seed', str(s)]
        for arm in ARMS:
            run(f'train_{pde}_s{s}_{arm}', ['exp72_utility.py', 'train', *a, '--arm', arm],
                lambda: os.path.exists(f(f'S_{pde}_s{s}_{arm}.pt')))
        run(f'evaluate_{pde}_s{s}', ['exp72_utility.py', 'evaluate', *a],
            lambda: os.path.exists(f(f'eval_{pde}_s{s}.json'))
            and all(os.path.getmtime(f(f'eval_{pde}_s{s}.json')) > os.path.getmtime(f(f'S_{pde}_s{s}_{arm}.pt'))
                    for arm in ARMS if os.path.exists(f(f'S_{pde}_s{s}_{arm}.pt'))))

In [ ]:
# 3. progress
import json
for pde in EQS:
    for s in SEEDS:
        have = [arm for arm in ARMS if os.path.exists(f(f'S_{pde}_s{s}_{arm}.pt'))]
        print(f'{pde} s{s}: steppers', ', '.join(have) or 'none')
        p = f(f'eval_{pde}_s{s}.json')
        if os.path.exists(p):
            for arm, r in json.load(open(p)).items():
                print(f'    {arm:8s} 16-step error: in {r["in"][1]:.2e}, out {r["out"][1]:.2e}')
for p in sorted(glob.glob(f('*.partial'))):
    print('in progress:', os.path.basename(p), time.ctime(os.path.getmtime(p)))

In [ ]:
# 4. copy the small outputs file by file to local disk (remount if Drive drops); fins_nb14.zip in /content and on Drive
local = '/content/fins_nb14'
shutil.rmtree(local, ignore_errors=True)
files = glob.glob(f('eval_*.json')) + glob.glob(f('S_*.json')) + glob.glob(os.path.join(RUNS, 'logs', '*.log'))
for p in sorted(set(files)):
    dst = os.path.join(local, os.path.relpath(p, R))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for attempt in range(3):
        try:
            shutil.copy2(p, dst)
            break
        except OSError as e:
            print('retry', p, e)
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)
out = '/content/fins_nb14.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(os.path.join(local, '**', '*'), recursive=True)):
        if os.path.isfile(p):
            z.write(p, os.path.relpath(p, local))
print(out, os.path.getsize(out), 'bytes')
try:
    shutil.copy2(out, '/content/drive/MyDrive/fins_nb14.zip')
    print('copied to MyDrive: fins_nb14.zip (upload this one to the chat)')
except OSError as e:
    print('Drive copy failed; download /content/fins_nb14.zip from the file pane:', e)